# Bring Your Own Schema

Skip schema discovery entirely. Define a Pydantic model, point catchfly at your documents, and get structured, normalized records back.

**What you'll learn:** User-provided schemas, auto field selection with custom schemas, `on_schema_ready` callback, JSON Schema dict input.

**Estimated API cost:** ~$0.02

In [ ]:
# pip install "catchfly[openai,export]"
# export OPENAI_API_KEY="sk-..."

## Define your schema

This example extracts structured data from SaaS support tickets. Pydantic `Field(description=...)` directly improves extraction quality — the LLM uses these descriptions.

In [ ]:
from pydantic import BaseModel, Field


class SupportTicket(BaseModel):
    subject: str = Field(description="Brief summary of the issue")
    category: str = Field(description="e.g., Authentication, Billing, Bug, Feature Request")
    priority: str = Field(description="Critical, High, Medium, or Low")
    product_area: str = Field(description="Which product area is affected")
    customer_email: str | None = Field(default=None, description="Customer contact email if mentioned")
    is_resolved: bool = Field(default=False, description="Whether the issue appears resolved")

## Load data and run

Even with a user-provided schema, `Pipeline.quick()` auto-selects which fields to normalize. The `StatisticalFieldSelector` looks at extracted values — it will pick `category` and `priority` (categorical, short values, repetitive) and skip `subject` (free-text, high cardinality) and `customer_email` (identifier pattern).

In [ ]:
from catchfly import Pipeline
from catchfly.demo import load_samples

docs = load_samples("support_tickets")

pipeline = Pipeline.quick(model="gpt-5.4-mini")

# No normalize_fields needed — StatisticalFieldSelector auto-detects categorical fields
results = pipeline.run(docs, schema=SupportTicket)

print(f"Extracted {len(results.records)} tickets")
print(f"Auto-normalized fields: {list(results.normalizations.keys())}")

## Inspect extracted records

In [ ]:
for ticket in results.records[:5]:
    email = f" ({ticket.customer_email})" if ticket.customer_email else ""
    print(f"[{ticket.priority}] {ticket.category}: {ticket.subject}{email}")

## Normalization: messy categories to clean taxonomy

The raw tickets use inconsistent category labels. The `StatisticalFieldSelector` identified these as categorical fields and `LLMCanonicalization` grouped the variants:

In [ ]:
cat_norm = results.normalizations.get("category")
if cat_norm:
    print("Category groups:")
    for canonical, members in cat_norm.clusters.items():
        print(f"  {canonical}: {members}")

pri_norm = results.normalizations.get("priority")
if pri_norm:
    print("\nPriority groups:")
    for canonical, members in pri_norm.clusters.items():
        print(f"  {canonical}: {members}")

## Schema callback: inspect before extraction

Use `on_schema_ready` to review (and optionally modify) the schema before extraction begins:

In [ ]:
## Explicit field override

Auto-selection is convenient, but you can always override it:

```python
# Only normalize category — skip everything else
results = pipeline.run(docs, schema=SupportTicket, normalize_fields=["category"])
```

## Schema callback: inspect before extraction

Use `on_schema_ready` to review (and optionally modify) the schema before extraction begins:

from catchfly._types import Schema


def review_schema(schema: Schema) -> Schema | None:
    print(f"Schema has {len(schema.json_schema['properties'])} fields:")
    for name in schema.json_schema["properties"]:
        print(f"  - {name}")
    # Return None to accept, or return a modified Schema to override
    return None


results = pipeline.run(
    docs,
    schema=SupportTicket,
    on_schema_ready=review_schema,
)

In [ ]:
json_schema = {
    "title": "SupportTicket",
    "type": "object",
    "properties": {
        "subject": {"type": "string", "description": "Brief summary"},
        "category": {"type": "string"},
        "priority": {"type": "string", "enum": ["Critical", "High", "Medium", "Low"]},
    },
    "required": ["subject", "category", "priority"],
}

results = pipeline.run(docs, schema=json_schema, normalize_fields=["category"])
print(f"Records: {len(results.records)}")

json_schema = {
    "title": "SupportTicket",
    "type": "object",
    "properties": {
        "subject": {"type": "string", "description": "Brief summary"},
        "category": {"type": "string"},
        "priority": {"type": "string", "enum": ["Critical", "High", "Medium", "Low"]},
    },
    "required": ["subject", "category", "priority"],
}

# Auto field selection works with JSON Schema input too
results = pipeline.run(docs, schema=json_schema)
print(f"Records: {len(results.records)}")
print(f"Auto-normalized: {list(results.normalizations.keys())}")

In [ ]:
df = results.to_dataframe()
df[["subject", "category", "priority"]]